In [21]:
import pandas as pd
from sqlalchemy import create_engine, inspect
from sqlalchemy.orm import sessionmaker
import matplotlib.pyplot as plt

In [22]:
# connect to the database
db_path = "/home/locascio/mydata/YServer/experiments/rchrono_comments.db"
engine = create_engine(f"sqlite:///{db_path}")

In [23]:
# Inspect Database Structure

inspector = inspect(engine)
table_names = inspector.get_table_names()
print("Tables in the database:")
print(table_names)

Tables in the database:
['article_topics', 'articles', 'emotions', 'follow', 'hashtags', 'images', 'interests', 'mentions', 'post', 'post_emotions', 'post_hashtags', 'post_topics', 'reactions', 'recommendations', 'rounds', 'user_interest', 'user_mgmt', 'voting', 'websites']


In [ ]:
if "mentions" in table_names:
    # Load the "follow" table into a Pandas DataFrame using a direct SQL query
    mentions_df = pd.read_sql_query("SELECT mentions.user_id AS mentioned, post.user_id AS mentioner, mentions.round AS round, post.round as post_round FROM mentions JOIN post ON mentions.post_id = post.id", engine)
else:
    raise ValueError("table not found in the database.")

print("First 5 rows:")
print(mentions_df.head())

# Basic statistics
print("Summary statistics of numeric columns:")
print(mentions_df.describe())

First 5 rows:
   mentioned  mentioner  round  post_round
0        419        164      1           1
1        419        233      2           2
2        164        233      2           2
3         15        233      2           2
4        233        329      3           3
Summary statistics of numeric columns:
         mentioned    mentioner        round   post_round
count  5630.000000  5630.000000  5630.000000  5630.000000
mean    253.600888   251.765187   357.125222   357.125222
std     142.161497   144.095717   204.991088   204.991088
min       1.000000     1.000000     1.000000     1.000000
25%     132.000000   129.000000   178.000000   178.000000
50%     261.000000   256.000000   353.000000   353.000000
75%     376.750000   377.000000   520.000000   520.000000
max     498.000000   498.000000   720.000000   720.000000


## Check round equality (Mentions - Posts)

In [25]:
mentions_df[mentions_df['round'] != mentions_df['post_round']]

,mentioned,mentioner,round,post_round


In [26]:
# Drop post_round column
mentions_df = mentions_df.drop(columns=['post_round'])

In [27]:
df = mentions_df
steps = 24

In [28]:
# Create a grouping column by dividing the round by 24
df['group'] = (df['round'] - 1) // steps

# Group by the new column and process the data into separate columns
grouped = df.groupby('group').apply(lambda group: pd.Series({
    'adjacency_matrix': group[['mentioner', 'mentioned']].to_numpy(),
    #'round': group['round'].tolist()
}))

# Reset index for readability (optional)
grouped.reset_index(drop=True, inplace=True)
# Display grouped data
grouped

/tmp/ipykernel_2358591/2769204968.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = df.groupby('group').apply(lambda group: pd.Series({


,adjacency_matrix
0,"[[164, 419], [233, 419], [233, 164], [233, 15]..."
1,"[[171, 377], [171, 441], [34, 474], [426, 147]..."
2,"[[138, 329], [138, 287], [398, 228], [398, 54]..."
3,"[[262, 214], [262, 140], [331, 31], [100, 214]..."
4,"[[54, 265], [54, 265], [54, 94], [27, 422], [2..."
5,"[[401, 277], [1, 128], [1, 392], [1, 85], [1, ..."
6,"[[487, 272], [487, 50], [487, 50], [487, 272],..."
7,"[[252, 423], [51, 423], [51, 252], [51, 236], ..."
8,"[[146, 441], [325, 180], [253, 208], [22, 249]..."
9,"[[228, 170], [228, 222], [228, 137], [248, 474..."
